# 09b · Detección de episodios fuera de muestra con el PCA (anexo R.b del Rmd final)

**Objetivo.** Anexo de robustez R.b (D15): detección de episodios fuera de muestra con el error
*walk-forward* del **PCA**, sin `torch`, con las **16 series** de D16 y los episodios del archivo
único `data/referencia/episodios_referencia.csv` (D7, D9). D15 ubica aquí la vía transformada,
porque es la que puede ser informativa en esta prueba; la vía sin transformar se calcula como
**contraste**, con el mismo código.

**Método** (`src/deteccion.py`, traducción de la parte PCA del notebook 09, sección 2). Para cada
año de evaluación $a$ = 2011, …, 2025 se estandariza con la media y la desviación (divisor $n$) de
los meses anteriores a enero de $a$, se ajusta un PCA de $d = 3$ componentes (D9) sobre esos meses
y se mide el error cuadrático medio de reconstrucción de los meses de $a$, dividido por el p95 del
error en entrenamiento. Discriminación (regla del 09): un mes es alarma si la razón es mayor que 1.

**Tolerancia.** La regla del 09 no tiene tolerancia (`tolerancia_meses = 0`). Como lectura propia
de esta tarea se informa también cada episodio ampliado ±3 meses (tolerancia principal de D7) y
±6 meses (sensibilidad); cada episodio se evalúa por separado, y un mes en dos episodios ampliados
cuenta en los dos. La crisis financiera global no puede evaluarse (el primer año es 2011).

**AE y LSTM** (D17.4): quedan en `\pendiente` con 16 series. Sus errores *walk-forward* con
17 series (`09_walkforward_errores.csv` del antecedente, sin reentrenar) se leen solo para
recalcular su discriminación con los episodios nuevos, rotulada como **antecedente, 17 series**.

**No regresión.** Con 17 series, `src/deteccion.error_walkforward_pca` reproduce la columna `pca` de
`09_walkforward_errores.csv` en las dos vías, y con tolerancia 0 la discriminación larga coincide
con `src/episodios.discriminacion_episodios` (criterio 8).

**Semillas.** Ninguna: el PCA es determinístico. **Salidas** (`data/results/final/anexo/`):
`09b_walkforward_pca.csv` y `09b_discriminacion_episodios.csv`.

In [1]:
# ---- Configuración y parámetros (adenda 3b, §3) ------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src.parametros import leer_parametros
from src.datos import cargar_vias
from src.deteccion import error_walkforward_pca, discriminacion_larga, ANIOS, D_PCA
from src.episodios import cargar_episodios, discriminacion_episodios

PARAM = leer_parametros(RAIZ)
EXCLUIDAS = PARAM['EXCLUIDAS']
DIR_ANEXO = PARAM['DIR']['transformadas']          # CARPETA/anexo
DIR_ANEXO.mkdir(parents=True, exist_ok=True)
X, Xt = cargar_vias(RAIZ, excluidas=EXCLUIDAS)
P = X.shape[1]
EPISODIOS = cargar_episodios(RAIZ)
TOLERANCIAS = (0, 3, 6)
ANTECEDENTE_WF = {'transformadas': RAIZ / 'data/results/09_walkforward_errores.csv',
                  'niveles': RAIZ / 'data/results/niveles/09_walkforward_errores.csv'}
print('EXCLUIDAS =', EXCLUIDAS, '| p =', P, '| salidas:', DIR_ANEXO)

EXCLUIDAS = ('Liquidez M2',) | p = 16 | salidas: /home/claude/tesis-modelamiento-matematico/data/results/final/anexo


## 1. No regresión con 17 series (solo lectura del antecedente)

In [2]:
X17, Xt17 = cargar_vias(RAIZ, excluidas=())
for via, M in [('transformadas', Xt17), ('niveles', X17)]:
    wf = error_walkforward_pca(M.to_numpy(), [f + '-01' for f in M.index])
    ref = pd.read_csv(ANTECEDENTE_WF[via], index_col=0, parse_dates=True)
    assert (wf.index == ref.index).all() and (wf.anio_modelo.to_numpy() == ref.anio_modelo.to_numpy()).all()
    dif = float(np.max(np.abs(wf.pca.to_numpy() - ref.pca.to_numpy())))
    assert dif < 1e-10, (via, dif)
    print(f'{via}: con 17 series reproduce la columna pca de {ANTECEDENTE_WF[via].name} (dif. máx. {dif:.1e})')

transformadas: con 17 series reproduce la columna pca de 09_walkforward_errores.csv (dif. máx. 1.5e-13)
niveles: con 17 series reproduce la columna pca de 09_walkforward_errores.csv (dif. máx. 1.5e-14)


## 2. Error walk-forward del PCA con las series vigentes

In [3]:
wf_vig = {}
for via, M in [('transformadas', Xt), ('niveles', X)]:
    wf_vig[via] = error_walkforward_pca(M.to_numpy(), [f + '-01' for f in M.index])
salida = pd.concat([w.reset_index().assign(via=v, series=P) for v, w in wf_vig.items()], ignore_index=True)
salida['fecha'] = salida.fecha.dt.strftime('%Y-%m')
salida = salida[['via', 'series', 'fecha', 'anio_modelo', 'pca']]
salida.to_csv(DIR_ANEXO / '09b_walkforward_pca.csv', index=False)
print(salida.groupby('via').pca.describe())

               count      mean       std       min       25%       50%  \
via                                                                      
niveles        180.0  1.532754  1.046131  0.365338  0.795879  1.135903   
transformadas  180.0  0.732056  3.043892  0.058611  0.183935  0.278918   

                    75%        max  
via                                 
niveles        1.914369   6.569368  
transformadas  0.450182  35.056336  


## 3. Discriminación de episodios

Filas: PCA con las series vigentes (transformadas, principal del anexo; niveles, contraste) y, como
antecedente con 17 series, PCA, AE y LSTM. Con tolerancia 0, la tabla larga reproduce la regla del
09 (`discriminacion_episodios`).

In [4]:
partes = []
for via in ('transformadas', 'niveles'):
    for tol in TOLERANCIAS:
        partes.append(discriminacion_larga(wf_vig[via], EPISODIOS, metodos=('pca',), tol=tol)
                      .assign(via=via, series=P, fuente='vigente (sin torch)'))
    ant = pd.read_csv(ANTECEDENTE_WF[via], index_col=0, parse_dates=True)
    for tol in TOLERANCIAS:
        partes.append(discriminacion_larga(ant, EPISODIOS, metodos=('pca', 'ae', 'lstm'), tol=tol)
                      .assign(via=via, series=17, fuente='antecedente (09_walkforward_errores.csv)'))
    # Con tolerancia 0, la regla del 09 (src.episodios) da las mismas cifras redondeadas
    for wf, ms, s in [(wf_vig[via], ('pca',), P), (ant, ('pca', 'ae', 'lstm'), 17)]:
        a = discriminacion_episodios(wf, EPISODIOS, metodos=ms)
        b = discriminacion_larga(wf, EPISODIOS, metodos=ms, tol=0)
        for r in b.itertuples():
            col = r.medida if r.episodio == '' else (f'mediana_{r.episodio}' if r.medida == 'mediana' else f'sensibilidad_{r.episodio}_%')
            assert abs(round(r.valor, 2) - a.loc[r.metodo, col]) < 1e-9, (via, s, r.metodo, col)
disc = pd.concat(partes, ignore_index=True)[['via', 'series', 'fuente', 'metodo', 'tolerancia_meses', 'medida', 'episodio', 'meses', 'valor']]
disc.to_csv(DIR_ANEXO / '09b_discriminacion_episodios.csv', index=False)
vista = disc[(disc.tolerancia_meses == 0) & (disc.medida.isin(['falsas_alarmas_%', 'sensibilidad_%']))]
display(vista.pivot_table(index=['via', 'series', 'metodo'], columns=['medida', 'episodio'], values='valor'))

medida                      falsas_alarmas_% sensibilidad_%  \
episodio                                           COVID-19   
via           series metodo                                   
niveles       16     pca           59.689922          100.0   
              17     ae            92.248062          100.0   
                     lstm          75.968992          100.0   
                     pca           59.689922          100.0   
transformadas 16     pca            3.875969          100.0   
              17     ae             3.875969          100.0   
                     lstm           3.100775          100.0   
                     pca            3.875969          100.0   

medida                                                      \
episodio                    Colapso de las materias primas   
via           series metodo                                  
niveles       16     pca                              95.0   
              17     ae                              100.0   
                     lstm                            100.0   
                     pca                              95.0   
transformadas 16     pca                               0.0   
              17     ae                                0.0   
                     lstm                              0.0   
                     pca                               0.0   

medida                                                      \
episodio                    Guerra comercial EE. UU.–China   
via           series metodo                                  
niveles       16     pca                               5.0   
              17     ae                               70.0   
                     lstm                            100.0   
                     pca                              10.0   
transformadas 16     pca                               0.0   
              17     ae                                0.0   
                     lstm                              0.0   
                     pca                               0.0   

medida                                                           
episodio                    Kuczynski / Odebrecht Taper tantrum  
via           series metodo                                      
niveles       16     pca                      0.0         100.0  
              17     ae                     100.0         100.0  
                     lstm                   100.0         100.0  
                     pca                      0.0         100.0  
transformadas 16     pca                    100.0           0.0  
              17     ae                     100.0           0.0  
                     lstm                     0.0           0.0  
                     pca                    100.0           0.0

**Comparación con R.** La hace el Rmd final (anexo R.b y anexo P) contra `R_anexo_walkforward_pca.csv`
y `R_anexo_discriminacion.csv`. **Lectura:** sin interpretación (regla 7).

In [5]:
for f_py, f_r, clave in [('09b_walkforward_pca.csv', 'R_anexo_walkforward_pca.csv', ['via', 'fecha']),
                         ('09b_discriminacion_episodios.csv', 'R_anexo_discriminacion.csv',
                          ['via', 'series', 'metodo', 'tolerancia_meses', 'medida', 'episodio'])]:
    ruta = DIR_ANEXO / f_r
    if not ruta.exists():
        print(f'aún no existe {f_r} (ejecutar el Rmd)')
        continue
    a = pd.read_csv(DIR_ANEXO / f_py, keep_default_na=False)
    b = pd.read_csv(ruta, keep_default_na=False)
    m = a.merge(b, on=clave, suffixes=('', '_r'))
    col = 'pca' if 'pca' in a.columns else 'valor'
    dif = float(np.max(np.abs(m[col] - m[col + '_r'])))
    assert len(m) == len(a) == len(b) and dif < 1e-10, (f_py, len(m), len(a), len(b), dif)
    print(f'{f_py}: coincide con {f_r} ({len(m)} filas, dif. máx. {dif:.1e})')

09b_walkforward_pca.csv: coincide con R_anexo_walkforward_pca.csv (360 filas, dif. máx. 4.3e-14)
09b_discriminacion_episodios.csv: coincide con R_anexo_discriminacion.csv (288 filas, dif. máx. 5.0e-14)
